# Acto 6 — Ontología OWL via Claude + MCP

**Este notebook NO simula a Claude.** Verifica que la infraestructura está lista:

1. El binario `nopaldb-mcp` existe y es ejecutable
2. La DB `biomedical_owl.db` tiene el esquema OWL cargado
3. El servidor registra los 12 tools incluyendo `classify_node`, `list_instances`, `list_subclasses`
4. Los 3 tools de ontología responden correctamente via JSON-RPC

La interacción real con Claude está en `docs/tutorial/acto_6_ontologia/README.md`.

## Prerrequisitos

- **Acto 3 completado:** ejecuta `03_biomedical_owl.ipynb` — la celda "0.5" genera
  `test_dbs/biomedical_owl.db` (requiere wheel con `--features python-owl`, incluido en `semantic`).
  Alternativa Rust:
  ```bash
  cargo run --example tutorial_acto_3_biomedical --no-default-features \\
    --features storage-sled,reasoner,owl-import,algorithms,analytics,hypergraph,ml \\
    -- test_dbs/biomedical_owl.db tutorials/data/biomedical.ttl
  ```
- **Acto 5 completado:** binario `nopaldb-mcp` compilado
- Ejecutar este notebook desde el directorio `tutorials/`

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

TUTORIALS_DIR = Path.cwd().resolve()
if TUTORIALS_DIR.name == "notebooks":
    TUTORIALS_DIR = TUTORIALS_DIR.parent

REPO_ROOT = TUTORIALS_DIR.parent
BINARY    = REPO_ROOT / "target" / "release" / "nopaldb-mcp"
DB_PATH   = TUTORIALS_DIR / "test_dbs" / "biomedical_owl.db"

print(f"Repo root : {REPO_ROOT}")
print(f"Binario   : {BINARY}")
print(f"DB        : {DB_PATH}")

## Paso 0 — Verificar binario y DB

In [ ]:
# Auto-generar la DB si no existe
if not DB_PATH.exists():
    print(f"DB no encontrada en {DB_PATH} — generando via biomedical_dataset...")
    import sys as _sys
    if str(TUTORIALS_DIR) not in _sys.path:
        _sys.path.insert(0, str(TUTORIALS_DIR))
    from shared.biomedical_dataset import generate as _gen
    _gen(DB_PATH)

assert BINARY.exists() and os.access(BINARY, os.X_OK), (
    f"Binario no encontrado en {BINARY}.\n"
    f"Ejecuta: cargo build -p nopaldb-mcp --release"
)
print(f"Binario OK: {BINARY} ({BINARY.stat().st_size // 1024} KB)")

assert DB_PATH.exists(), (
    f"DB no encontrada en {DB_PATH}.\n"
    f"Ejecuta primero el Acto 3 notebook (03_biomedical_owl.ipynb)."
)
print(f"DB OK: {DB_PATH}")

## Paso 1 — Verificar esquema OWL en la DB

In [ ]:
import gc
import nopaldb

g = nopaldb.Graph.open(str(DB_PATH))
try:
    res = g.execute_nql(
        "find n.label as clase, count(*) as total "
        "from (n) group by n.label order by total desc"
    )
    rows = getattr(res, 'query', res)
    labels = {r.get('clase', r.get('n.label')): r.get('total', r.get('count(*)')) for r in rows}
finally:
    g.close()
    del g
    gc.collect()  # libera el lock de Sled antes del siguiente subprocess

print("Clases en la DB:")
for label, cnt in sorted(labels.items(), key=lambda x: -x[1]):
    print(f"  {label:25s} {cnt}")

EXPECTED_LABELS = {"Disease", "Infection", "ViralInfection", "BacterialInfection",
                   "Treatment", "Antiviral", "Antibiotic"}
found = set(labels.keys())
missing = EXPECTED_LABELS - found
assert not missing, f"Faltan labels OWL: {missing}. Regenera con el Acto 3."
print("\nSchemaOWL OK — 7 clases verificadas")

## Paso 2 — Smoke test JSON-RPC: 12 tools registrados

In [ ]:
# Tools genéricos presentes en todas las ediciones.
# compute_risk solo existe en builds con la feature "risk" (no Community).
EXPECTED_TOOLS = {
    "graph_query", "schema_info", "get_node", "get_neighbors", "find_path",
    "run_pagerank", "similar_nodes", "schema_by_kind",
    "classify_node", "list_instances", "list_subclasses",
}
OPTIONAL_TOOLS = {"compute_risk"}

messages = "\n".join([
    json.dumps({
        "jsonrpc": "2.0", "id": 1, "method": "initialize",
        "params": {
            "protocolVersion": "2024-11-05",
            "capabilities": {},
            "clientInfo": {"name": "smoke-test", "version": "0"}
        }
    }),
    json.dumps({"jsonrpc": "2.0", "method": "notifications/initialized"}),
    json.dumps({"jsonrpc": "2.0", "id": 2, "method": "tools/list"}),
]) + "\n"

proc = subprocess.Popen(
    [str(BINARY), "--db", str(DB_PATH), "--readonly"],
    stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.PIPE,
)
stdout_bytes, stderr_bytes = proc.communicate(messages.encode(), timeout=15)

responses = [
    json.loads(line)
    for line in stdout_bytes.decode().strip().split("\n")
    if line.strip()
]

tools_resp = next((r for r in responses if r.get("id") == 2), None)
assert tools_resp is not None, f"No se recibió respuesta a tools/list. Stderr: {stderr_bytes.decode()[-500:]}"
assert "result" in tools_resp, f"tools/list retornó error: {tools_resp}"

tools = {t["name"]: t for t in tools_resp["result"]["tools"]}
print(f"Tools registrados ({len(tools)}):")
for name in sorted(tools):
    mark = "[ONT]" if name in {"classify_node", "list_instances", "list_subclasses"} else "     "
    print(f"  {mark} {name}")

missing = EXPECTED_TOOLS - set(tools.keys())
assert not missing, f"Tools faltantes: {missing}"
extra = set(tools.keys()) - EXPECTED_TOOLS - OPTIONAL_TOOLS
assert not extra, f"Tools inesperados: {extra}"
print(f"\nSMOKE TEST OK — {len(tools)} tools registrados (incluyendo 3 de ontología)")

## Paso 3 — Gate: classify_node (Covid19 ∈ Disease)

In [ ]:
def mcp_call(tool_name, arguments):
    """Envía una llamada tools/call al servidor MCP y retorna el profile parseado."""
    msgs = "\n".join([
        json.dumps({"jsonrpc": "2.0", "id": 1, "method": "initialize",
                    "params": {"protocolVersion": "2024-11-05", "capabilities": {},
                               "clientInfo": {"name": "gate", "version": "0"}}}),
        json.dumps({"jsonrpc": "2.0", "method": "notifications/initialized"}),
        json.dumps({"jsonrpc": "2.0", "id": 2, "method": "tools/call",
                    "params": {"name": tool_name, "arguments": arguments}}),
    ]) + "\n"
    proc = subprocess.Popen(
        [str(BINARY), "--db", str(DB_PATH), "--readonly"],
        stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.PIPE,
    )
    stdout, _ = proc.communicate(msgs.encode(), timeout=30)
    responses = [json.loads(l) for l in stdout.decode().strip().split("\n") if l.strip()]
    resp = next((r for r in responses if r.get("id") == 2), None)
    assert resp and "result" in resp, f"Tool call falló: {resp}"
    content = resp["result"]["content"]
    return json.loads(content[0]["text"]) if content else {}

# classify_node: Covid19 debe ser instancia de Disease (transitivamente)
result = mcp_call("classify_node", {"name": "Covid19", "class": "Disease"})
print(f"classify_node(Covid19, Disease):")
print(f"  is_instance : {result.get('is_instance')}")
print(f"  explanation : {result.get('explanation')}")

assert result.get("is_instance") is True, (
    f"GATE FALLIDO: Covid19 debería ser instancia de Disease (transitiva).\n"
    f"  Verifica que el Acto 3 cargó la jerarquía OWL correctamente.\n"
    f"  Result: {json.dumps(result, indent=2)}"
)

# Negativo: Tuberculosis NO es ViralInfection
result_neg = mcp_call("classify_node", {"name": "Tuberculosis", "class": "ViralInfection"})
assert result_neg.get("is_instance") is False, (
    f"GATE FALLIDO: Tuberculosis NO debería ser ViralInfection.\nResult: {result_neg}"
)
print(f"\nclassify_node(Tuberculosis, ViralInfection): is_instance={result_neg.get('is_instance')} ✓")
print("\nGATE OK — classify_node funciona (positivo transitivo + negativo)")

## Paso 4 — Gate: list_instances(Disease) retorna 5 individuos

In [ ]:
result = mcp_call("list_instances", {"class": "Disease", "limit": 20})
instances = result.get("instances", [])
names = [i["name"] for i in instances]

print(f"list_instances(Disease) → {result.get('count')} individuos:")
for inst in instances:
    print(f"  {inst['name']:20s}  ({inst['label']})")

EXPECTED_INDIVIDUALS = {"Covid19", "Influenza", "Dengue", "Tuberculosis", "StrepThroat"}
found = set(names)
missing = EXPECTED_INDIVIDUALS - found
assert not missing, (
    f"GATE FALLIDO: faltan individuos en list_instances(Disease): {missing}\n"
    f"  Nota: instanceOf es transitivo — deben incluirse ViralInfection y BacterialInfection.\n"
    f"  Result: {json.dumps(result, indent=2)}"
)
assert len(instances) == 5, f"Se esperaban 5 instancias de Disease, se recibieron {len(instances)}"
print("\nGATE OK — list_instances(Disease) retorna los 5 individuos transitivos")

## Paso 5 — Gate: list_subclasses(Disease)

In [ ]:
result = mcp_call("list_subclasses", {"class": "Disease"})

print(f"list_subclasses(Disease):")
print(f"  Directas   : {result.get('direct_subclasses')}")
print(f"  Indirectas : {result.get('indirect_subclasses')}")
print(f"  Total      : {result.get('total')}")

assert "Infection" in result.get("direct_subclasses", []), (
    f"GATE FALLIDO: 'Infection' debe ser subclase directa de 'Disease'.\nResult: {result}"
)
indirect = set(result.get("indirect_subclasses", []))
assert "ViralInfection" in indirect and "BacterialInfection" in indirect, (
    f"GATE FALLIDO: ViralInfection y BacterialInfection deben ser subclases indirectas.\nResult: {result}"
)
print("\nGATE OK — list_subclasses(Disease) mapea la jerarquía completa")

---

## Infraestructura lista — abre Claude

Si los 5 pasos anteriores pasaron:

1. Configura Claude Desktop con `biomedical_owl.db` (ver README del Acto 6)
2. Sigue los Pasos 1-6 del README

---

| Tool MCP | Paso en README | Lo que Claude hace |
|----------|---------------|--------------------|
| `schema_by_kind` | 1 | Descubre el vocabulario OWL |
| `list_subclasses` | 2 | Mapea la jerarquía de herencia |
| `classify_node` | 3 | Verifica membership transitivo |
| `list_instances` | 4-5 | Enumera individuos de una superclase |
| `graph_query` | 6 | NQL con `instanceOf` en WHERE |

**Siguiente:** `docs/MCP_SERVER_IMPL.md` para el contrato completo de los 12 tools.